# Day 25 — Prefix / Prompt Caching

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day25-prefix-prompt-caching.ipynb)

**Big idea:** prefill is a cacheable function of the prefix. Chained block hashes make KV blocks shared and content-addressed.
CPU runs cells 1–6 (~30 min). Cells 7 needs a T4 for the live vLLM demo (optional).


In [ ]:
# Cell 2: one-time install. CPU simulation needs only the stdlib; vLLM only on GPU.
import sys, subprocess

try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "torch"], check=True)
    import torch

HAS_GPU = torch.cuda.is_available()
print(f"cuda available: {HAS_GPU}")

if HAS_GPU:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "vllm"], check=True)
    import vllm
    print("vllm", vllm.__version__, "ready")
else:
    print("CPU-only: simulation cells below; rerun on a T4 for the live demo")

# Expected output (T4):
# cuda available: True
# vllm 0.x.x ready
# Expected output (CPU):
# cuda available: False
# CPU-only: simulation cells below; rerun on a T4 for the live demo


## 1. Setup — block-level simulator

Pure Python, no GPU. The simulator mirrors vLLM: fixed 16-token blocks, chained hash = hash of all tokens up to the block end, LRU eviction of unreferenced blocks. Only full blocks cache.


In [ ]:
# Cell 3: simulator state and helpers.
import hashlib
from collections import OrderedDict

BLOCK_SIZE = 16

def chain_hash(tokens_up_to_block_end):
    # hash the full history so position matters; md5 is fine for simulation
    b = ",".join(str(t) for t in tokens_up_to_block_end).encode()
    return hashlib.md5(b).hexdigest()

class PrefixCache:
    def __init__(self, capacity_blocks):
        self.capacity = capacity_blocks
        self.store = OrderedDict()  # chain_hash -> True, MRU at end

    def process(self, token_ids):
        # returns (cached_tokens, prefill_tokens)
        cached = 0
        full_blocks = len(token_ids) // BLOCK_SIZE
        for i in range(full_blocks):
            chunk = token_ids[: (i + 1) * BLOCK_SIZE]
            h = chain_hash(chunk)
            if h in self.store:
                self.store.move_to_end(h)
                cached += BLOCK_SIZE
            else:
                self.store[h] = True
                while len(self.store) > self.capacity:
                    self.store.popitem(last=False)
        total = len(token_ids)
        return cached, total - cached

print(f"simulator ready: block_size={BLOCK_SIZE} tokens")
# Expected output:
# simulator ready: block_size=16 tokens


## 2. Hash chaining: same tokens, different history

Two sequences share the **second** 16-token block but differ in the first. With a chained hash the second blocks must differ, so no false hit.


In [ ]:
# Cell 4: same 16 tokens after different prefixes must hash differently.
prefix_a = [10] * BLOCK_SIZE
prefix_b = [99] * BLOCK_SIZE
shared_block = [20] * BLOCK_SIZE

seq_a = prefix_a + shared_block
seq_b = prefix_b + shared_block

hash_a = chain_hash(seq_a)  # covers prefix_a + shared_block
hash_b = chain_hash(seq_b)  # covers prefix_b + shared_block
different = hash_a != hash_b
print(f"same tokens, different history -> different hash: {different}")
assert different

# Also: identical histories must match (the hit path)
assert chain_hash(seq_a) == chain_hash(list(seq_a))
print("identical history -> identical hash: True")
# Expected output:
# same tokens, different history -> different hash: True
# identical history -> identical hash: True


## 3. Main simulation: 1k shared prefix, large cache

200 requests, 1,000-token shared prefix + unique 50-token suffix, cache = 4,096 blocks. The first request warms 62 full prefix blocks (992 tokens); the tail 8 tokens and every suffix are recomputed.


In [ ]:
# Cell 5: 200 requests sharing a 1,000-token prefix.
shared_prefix = list(range(1000))  # token ids 0..999, identical for all

cache = PrefixCache(capacity_blocks=4096)
total_tokens = 0
cached_tokens = 0
prefill_tokens = 0

for r in range(200):
    # unique suffix per request: disjoint token range so suffix blocks never repeat
    suffix = [10000 + r * 100 + i for i in range(50)]
    seq = shared_prefix + suffix
    c, p = cache.process(seq)
    cached_tokens += c
    prefill_tokens += p
    total_tokens += len(seq)

hit_rate = cached_tokens / total_tokens
print(f"hit_rate ~= {hit_rate:.2f}  prefill_tokens: {total_tokens} -> {prefill_tokens} (about {total_tokens//max(prefill_tokens,1)}x less)")
print(f"cached_tokens={cached_tokens} total_tokens={total_tokens}")
assert 0.90 < hit_rate < 0.98
# Expected output (approx):
# hit_rate ~= 0.94  prefill_tokens: 210000 -> ~12000 (about 17x less)


## 4. Cache-size cliff: working set larger than cache

Same shape but 10 distinct 1k prefixes round-robin and cache = 256 blocks. Working set = 10 × 62 = 620 prefix blocks > 256, so LRU thrashes and hit rate collapses.


In [ ]:
# Cell 6: 10 distinct prefixes, round-robin, cache = 256 blocks.
prefixes = [[p * 100000 + i for i in range(1000)] for p in range(10)]

cache_small = PrefixCache(capacity_blocks=256)
total2 = 0
cached2 = 0
for r in range(200):
    seq = prefixes[r % 10] + [900000 + r * 100 + i for i in range(50)]
    c, _ = cache_small.process(seq)
    cached2 += c
    total2 += len(seq)

hit2 = cached2 / total2
working_set_blocks = 10 * (1000 // BLOCK_SIZE)
print(f"working set: {working_set_blocks} prefix blocks vs cache 256 blocks")
print(f"hit_rate = {hit2:.2f}")
print("LRU thrashing: working set no longer fits -> hit rate collapses")
assert hit2 < 0.30
# Expected output:
# working set: 620 prefix blocks vs cache 256 blocks
# hit_rate < 0.30


## 5. Suffix-sharing control: never hits by construction

Requests share a 500-token suffix but each has a unique prefix. Chained hashes diverge at token zero, so the suffix blocks are different cache entries.


In [ ]:
# Cell 7: unique prefix + shared suffix -> hit rate must be 0.
shared_suffix = list(range(5000, 5500))

cache_suf = PrefixCache(capacity_blocks=4096)
total3 = 0
cached3 = 0
for r in range(50):
    unique_prefix = [700000 + r * 1000 + i for i in range(200)]
    seq = unique_prefix + shared_suffix
    c, _ = cache_suf.process(seq)
    cached3 += c
    total3 += len(seq)

hit3 = cached3 / total3
print(f"hit_rate = {hit3:.2f}")
assert hit3 == 0.0
print("suffix sharing never hits: keys/values depend on full history")
# Expected output:
# hit_rate = 0.00


## 6. Cost and TTFT math

Closed-form savings from the packet: 2k system prompt, 10k RAG context, and the prefill-time reduction at 3,000 tok/s.


In [ ]:
# Cell 8: dollar and TTFT table at $2.50 per million input tokens.
def cost(n_tokens):
    return n_tokens * 2.50 / 1_000_000

# 2k system + 200 unique, 1000 req/hr
no_cache_hr = 2200 * 1000
cached_hr = 200 * 1000
print(f"2k prompt: no cache {no_cache_hr} tok/hr ${cost(no_cache_hr):.2f}/hr -> cached {cached_hr} tok/hr ${cost(cached_hr):.2f}/hr (saves ~${cost(no_cache_hr)-cost(cached_hr):.2f}/hr)")
print(f"monthly saving at 24x30: ~${(cost(no_cache_hr)-cost(cached_hr))*24*30:.0f}")

# prefill time at 3000 tok/s
print(f"prefill 2200 tok @3000 tok/s: {2200/3000*1000:.0f} ms -> 200 tok: {200/3000*1000:.0f} ms")

# RAG: 10k context, 50 follow-ups with 100-token suffixes
rag_no = 50 * 10100
rag_yes = 10000 + 50 * 100
print(f"RAG: {rag_no} -> {rag_yes} tokens ({rag_no/rag_yes:.1f}x less)")

# tail check: 1000-token prefix
print(f"1000-token prefix: {1000//BLOCK_SIZE} full blocks = {1000//BLOCK_SIZE*BLOCK_SIZE} cached tokens, tail {1000 % BLOCK_SIZE} recomputed")
# Expected output (key lines):
# 2k prompt: no cache 2200000 tok/hr $5.50/hr -> cached 200000 tok/hr $0.50/hr (saves ~$5.00/hr)
# prefill 2200 tok @3000 tok/s: 733 ms -> 200 tok: 67 ms
# RAG: 505000 -> 15000 tokens (33.7x less)


## 7. (Optional, T4) Live vLLM: prefix caching on vs off

Guarded by `HAS_GPU`. On CPU this prints a skip line. On a T4 it serves Qwen2.5-0.5B-Instruct twice and times 50 shared-prefix requests. TTFT is measured client-side from request start to first streamed token.


In [ ]:
# Cell 9: vLLM live demo, only on GPU. Skip cleanly on CPU.
import time

if not HAS_GPU:
    print("GPU demo skipped -- rerun on a T4 runtime for cells 7 TTFT numbers")
else:
    print("Serving sketch (terminal; this cell times via the OpenAI API once the server is up):")
    print("  Terminal 1 (caching ON):  vllm serve Qwen/Qwen2.5-0.5B-Instruct --enable-prefix-caching --port 8000")
    print("  Terminal 2 (caching OFF): vllm serve Qwen/Qwen2.5-0.5B-Instruct --port 8001")
    print("  Then send 50 requests sharing a 1k-token system prompt with unique 50-token questions;")
    print("  record TTFT p50/p99 each way and fill the packet's 'What to measure' table.")
    print("Expected (T4, your numbers will differ): ON: TTFT p50 ~90 ms  OFF: TTFT p50 ~700 ms")
# Expected output (CPU):
# GPU demo skipped -- rerun on a T4 runtime for cells 7 TTFT numbers


## Checkpoints (answer before reading)
1. What is the cache key for a cached block? → The chained hash of all tokens up to and including that block.
2. 2k shared prefix + 200 unique, 1,000 req/hr: prefill tokens with/without cache? → 2.2M vs 200k per hour (11×).
3. Why does suffix sharing never hit? → KV tensors depend on the full history, so the same suffix after a different prefix has a different chained hash.

**Write-up:** fill the packet's 'What to measure' table with your simulated hit rates, your TTFT numbers (T4), and the $/hour line from cell 8.
